# Calibration: six reader conditions from saved compression
Attach **this repository directory** (including `data/benchmark/calibration`) and **itercomp_runs_export_calibration.zip** as Kaggle inputs. Kaggle may automatically unpack the ZIP; the extracted dataset folder is also supported. Enable GPU + Internet and enable the approved account's **HF_TOKEN** secret for this notebook. Run cells in order.

This notebook runs **Raw / Oracle / saved IterCOMP × current prompt / current prompt with only the abstention instruction removed**, on the same 150 calibration questions. It reuses the IterCOMP compression bytes; it never runs BGE or the controller. Oracle explicitly uses gold supporting paragraph IDs to construct its context. Gold answers are used only for scoring.

All six conditions run fresh reader inference, including the original prompt as a reproducibility control. Quantization, checkpoint revision, generation settings, document ordering, and title handling are held fixed. Each condition runs in its own subprocess to release GPU memory. Interrupted reader conditions resume by ID; sample-level failures remain visible and are not silently retried. No inference was performed while creating this notebook.


In [ ]:
from pathlib import Path, PurePosixPath
import os, sys, json, hashlib, shutil, subprocess, csv, copy, statistics
from zipfile import ZipFile

os.environ.update(USE_TORCH='1', USE_TF='0', USE_FLAX='0')
# Leave None to discover a unique repository and ZIP or extracted export automatically.
SOURCE_REPO = None  # e.g. Path('/kaggle/input/datasets/your-name/your-project/musique_itercomp')
CALIBRATION_ZIP = None  # optional path to an actual ZIP file
CALIBRATION_DIR = None  # optional extracted export folder; choose this OR CALIBRATION_ZIP
DATA_DIR = None  # optional override for the prepared benchmark directory
INPUT_ROOT = Path('/kaggle/input')
REPO = Path('/kaggle/working/musique_itercomp_reader_ablation')
OUTPUT_ROOT = Path('/kaggle/working/itercomp_reader_calibration')
READER_DEVICE = 'cuda:0'
INSTALL_DEPENDENCIES = True  # use False only when this exact stack is installed already

def file_sha(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024*1024), b''):
            digest.update(block)
    return digest.hexdigest()

def read_jsonl(path):
    with Path(path).open(encoding='utf-8') as handle:
        return [json.loads(line) for line in handle if line.strip()]

def write_once(path, text):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        if path.read_text(encoding='utf-8') != text:
            raise ValueError(f'Existing experiment file differs: {path}. Choose a fresh OUTPUT_ROOT.')
    else:
        path.write_text(text, encoding='utf-8')

def save_json(path, value):
    write_once(path, json.dumps(value, ensure_ascii=False, indent=2, sort_keys=True) + '\n')

def choose_path(explicit, candidates, label):
    if explicit is not None:
        return Path(explicit).resolve()
    candidates = sorted(set(Path(p).resolve() for p in candidates))
    if len(candidates) != 1:
        raise ValueError(f'Set {label} explicitly; found {len(candidates)} candidates: {candidates}')
    return candidates[0]


## Locate inputs and create a working copy
Kaggle often unpacks uploaded ZIPs automatically. Auto-discovery supports an actual ZIP **or an extracted export folder**, using calibration metadata and the input hash rather than the dataset's display name. If several copies are attached, set `CALIBRATION_DIR` (extracted folder) or `CALIBRATION_ZIP` (file) explicitly. An extracted folder must contain both the calibration compression and reader run directories. Prepared splits are never regenerated.


In [ ]:
SOURCE_REPO = choose_path(SOURCE_REPO, [
    p.parent for p in INPUT_ROOT.rglob('pyproject.toml')
    if (p.parent/'src/itercomp/artifacts.py').is_file()
    and (p.parent/'scripts/run_experiment.py').is_file()
], 'SOURCE_REPO')
DATA_DIR = Path(DATA_DIR) if DATA_DIR is not None else SOURCE_REPO/'data/benchmark'
INPUTS = DATA_DIR/'calibration/inputs.jsonl'
GOLD = DATA_DIR/'calibration/gold.jsonl'
for required in (SOURCE_REPO/'pyproject.toml', INPUTS, GOLD):
    if not required.is_file():
        raise FileNotFoundError(required)

if CALIBRATION_DIR is not None and CALIBRATION_ZIP is not None:
    raise ValueError('Set only CALIBRATION_DIR or CALIBRATION_ZIP, not both')
explicit_source = CALIBRATION_DIR if CALIBRATION_DIR is not None else CALIBRATION_ZIP
source_candidates = []
if explicit_source is None:
    source_candidates.extend(INPUT_ROOT.rglob('itercomp_runs_export_calibration.zip'))
    expected_dataset = file_sha(INPUTS)
    for metadata_path in INPUT_ROOT.rglob('run_metadata.json'):
        try:
            metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
        except (OSError, ValueError):
            continue
        identity = metadata.get('identity', {})
        if (identity.get('stage') == 'compress' and identity.get('method') == 'itercomp'
                and identity.get('dataset_sha256') == expected_dataset
                and (metadata_path.parent/'compression_results.jsonl').is_file()):
            # The parent of the compression run contains its sibling reader run.
            source_candidates.append(metadata_path.parent.parent)
calibration_source = choose_path(explicit_source, source_candidates, 'CALIBRATION_DIR or CALIBRATION_ZIP')
if not calibration_source.exists():
    raise FileNotFoundError(calibration_source)

if not REPO.exists():
    shutil.copytree(SOURCE_REPO, REPO, ignore=shutil.ignore_patterns(
        '.git', '.venv', 'venv', '__pycache__', 'data', 'results', 'analysis', '*.zip'))
elif not (REPO/'src/itercomp/artifacts.py').is_file():
    raise ValueError(f'{REPO} is not an IterCOMP working copy')
os.chdir(REPO)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
directory_sha = None
if calibration_source.is_dir():
    EXTRACTED = calibration_source
    archive_sha = None  # Original ZIP bytes are unavailable after Kaggle extraction.
    directory_files = [
        [path.relative_to(EXTRACTED).as_posix(), file_sha(path)]
        for path in sorted(EXTRACTED.rglob('*')) if path.is_file()
    ]
    directory_sha = hashlib.sha256(json.dumps(
        directory_files, ensure_ascii=False, separators=(',', ':')).encode('utf-8')).hexdigest()
    source_info = {'format': 'extracted_directory', 'directory_sha256': directory_sha,
                   'files': directory_files}
else:
    if CALIBRATION_DIR is not None:
        raise ValueError('CALIBRATION_DIR must point to an extracted directory')
    archive_sha = file_sha(calibration_source)
    EXTRACTED = OUTPUT_ROOT.parent / ('calibration_export_' + archive_sha[:16])
    if not EXTRACTED.exists():
        with ZipFile(calibration_source) as archive:
            for member in archive.infolist():
                parts = PurePosixPath(member.filename.replace('\\', '/')).parts
                if (not parts or PurePosixPath(*parts).is_absolute()
                        or '..' in parts or any(':' in part for part in parts)
                        or (member.external_attr >> 16) & 0o170000 == 0o120000):
                    raise ValueError(f'Unsafe archive member: {member.filename}')
            archive.extractall(EXTRACTED)
        (EXTRACTED/'.source_archive_sha256').write_text(archive_sha, encoding='utf-8')
    elif (EXTRACTED/'.source_archive_sha256').read_text(encoding='utf-8') != archive_sha:
        raise ValueError('Extraction cache does not match the archive')
    source_info = {'format': 'zip', 'archive_sha256': archive_sha}
save_json(OUTPUT_ROOT/'provenance/calibration_source.json', source_info)
print('Repository:', SOURCE_REPO)
print('Calibration source:', calibration_source, '| format:', source_info['format'])
print('Outputs:', OUTPUT_ROOT)


## Install the existing GPU stack
The repository requirements retain Kaggle's CUDA PyTorch. A fresh runtime is preferable if other model libraries have already been imported. The explicit `src` path also makes the editable package visible in the current notebook kernel.


In [ ]:
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO/'requirements-gpu.txt')], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(REPO), '--no-deps'], check=True)
sys.path.insert(0, str(REPO/'src'))
sys.path.insert(0, str(REPO))
import importlib
importlib.invalidate_caches()
from itercomp.config import config_from_dict
from itercomp.artifacts import inspect_artifacts, source_hash
from itercomp.dataset import load_inputs
from itercomp.prompt_builder import load_prompts, prompt_hashes
import yaml


## Select and verify the full calibration compression
The archive also contains smoke results. Selection below requires the **calibration input hash, full ID set, completed stage, and IterCOMP method**. Artifact checks verify source/config/prompt identity, hashes, contexts, and provenance before any weights are loaded. A mismatch must be resolved with the matching repository/configuration; do not edit hashes to bypass it.


In [ ]:
dataset_sha = file_sha(INPUTS)
samples = load_inputs(INPUTS)
sample_ids = [sample.id for sample in samples]
if len(sample_ids) != 150 or len(set(sample_ids)) != len(sample_ids):
    raise ValueError('Expected the existing complete 150-question calibration split')
candidates = []
for path in EXTRACTED.rglob('run_metadata.json'):
    meta = json.loads(path.read_text(encoding='utf-8'))
    identity = meta.get('identity', {})
    if (identity.get('stage') == 'compress' and identity.get('method') == 'itercomp'
            and identity.get('dataset_sha256') == dataset_sha
            and set(identity.get('selected_ids', [])) == set(sample_ids)
            and meta.get('selected_subset_complete')
            and meta.get('status') in ('complete', 'partial')):
        candidates.append((path.parent/'compression_results.jsonl', meta))
if len(candidates) != 1:
    raise ValueError(f'Expected one matching full calibration compression; found {len(candidates)}')
COMPRESSION_RESULTS, compression_meta = candidates[0]
base_data = copy.deepcopy(compression_meta['details']['config'])
base_cfg = config_from_dict(base_data)
expected_source = compression_meta['identity']['compression_contract']['algorithm_source_sha256']
if source_hash() != expected_source:
    raise ValueError('Attached repository source does not match saved compression. Attach the matching directory.')
original_prompts = load_prompts(base_cfg)
inspect_artifacts(COMPRESSION_RESULTS, samples, base_cfg, dataset_sha, prompt_hashes(original_prompts))
compression_errors = [row['id'] for row in read_jsonl(COMPRESSION_RESULTS) if row.get('error')]
if compression_errors:
    raise ValueError(f'Calibration compression contains errors: {compression_errors}')

# Find the archived reader paired with these exact compression bytes.
archived_readers = []
for path in EXTRACTED.rglob('run_metadata.json'):
    meta = json.loads(path.read_text(encoding='utf-8'))
    identity = meta.get('identity', {})
    if (identity.get('stage') == 'read' and identity.get('dataset_sha256') == dataset_sha
            and identity.get('compression_file_sha256') == file_sha(COMPRESSION_RESULTS)
            and set(identity.get('selected_ids', [])) == set(sample_ids)
            and meta.get('selected_subset_complete')):
        archived_readers.append((path.parent/'predictions.jsonl', meta))
if len(archived_readers) != 1:
    raise ValueError('Expected one archived reader run paired with the calibration compression')
ARCHIVED_PREDICTIONS, archived_reader_meta = archived_readers[0]
if file_sha(ARCHIVED_PREDICTIONS) != archived_reader_meta['output_sha256']:
    raise ValueError('Archived reader checksum mismatch')
if archived_reader_meta['identity']['reader_prompt_sha256'] != prompt_hashes(original_prompts)['reader']:
    raise ValueError('Current reader prompt does not match the calibration prompt')
# Use the reader configuration actually used by the archived reader stage.
base_data['reader'] = copy.deepcopy(archived_reader_meta['identity']['reader'])
reader_init = archived_reader_meta['component_initialization']['reader']
reader_revision = reader_init.get('resolved_revision')
if not reader_revision:
    raise ValueError('Archived reader revision is unavailable; identify it before comparing conditions')
base_data['reader']['revision'] = reader_revision
base_data['reader']['tokenizer_revision'] = reader_init.get('resolved_tokenizer_revision') or reader_revision
base_data['reader']['device'] = READER_DEVICE
if base_data['reader']['backend'] != 'transformers' or base_data['reader']['do_sample']:
    raise ValueError('This notebook expects the archived deterministic Transformers reader')
if set(row['id'] for row in read_jsonl(GOLD)) != set(sample_ids):
    raise ValueError('Calibration gold IDs differ from inputs')
print('Verified calibration samples:', len(sample_ids), '| compression errors:', len(compression_errors))
print('Reader:', base_data['reader']['model_id'], '| pinned revision:', reader_revision)
save_json(OUTPUT_ROOT/'provenance/compression_metadata.json', compression_meta)
save_json(OUTPUT_ROOT/'provenance/archived_reader_metadata.json', archived_reader_meta)


## Define the two prompts and six conditions
The second prompt removes exactly one sentence: `If the information is insufficient, return exactly INSUFFICIENT_INFORMATION.` No request to guess, extra reasoning instruction, or other prompt edit is added.

The original reader parser is retained. A model may still voluntarily return the sentinel or an empty response; both are counted as abstentions. The two prompt files live outside the source package so compression source identity is preserved.


In [ ]:
abstention_instruction = ' If the information is insufficient, return exactly INSUFFICIENT_INFORMATION.'
current_prompt = original_prompts['reader']
if current_prompt.count(abstention_instruction) != 1:
    raise ValueError('Expected exactly one original abstention instruction')
prompt_texts = {
    'current': current_prompt,
    'no_abstention_instruction': current_prompt.replace(abstention_instruction, '', 1),
}
CONFIGS = {}
for name, text in prompt_texts.items():
    prompt_path = OUTPUT_ROOT/'prompts'/f'reader_{name}.txt'
    write_once(prompt_path, text)
    data = copy.deepcopy(base_data)
    data['prompts']['reader'] = str(prompt_path)
    cfg = config_from_dict(data)
    # Reader-only settings may change; the compression contract must still match.
    inspect_artifacts(COMPRESSION_RESULTS, samples, cfg, dataset_sha, prompt_hashes(load_prompts(cfg)))
    config_path = OUTPUT_ROOT/'configs'/f'{name}.yaml'
    write_once(config_path, yaml.safe_dump(data, sort_keys=False, allow_unicode=True))
    CONFIGS[name] = config_path
JOBS = [
    {'name': f'{context}_{prompt}', 'context': context, 'prompt': prompt,
     'config': CONFIGS[prompt], 'output': OUTPUT_ROOT/'readers'/f'{context}_{prompt}'}
    for context in ('raw', 'oracle', 'itercomp') for prompt in prompt_texts
]
manifest = {
    'experiment': 'calibration_reader_3_contexts_2_prompts_v1',
    'n': len(sample_ids), 'sample_ids': sample_ids, 'archive_sha256': archive_sha,
    'inputs_sha256': dataset_sha, 'gold_sha256': file_sha(GOLD),
    'compression_sha256': file_sha(COMPRESSION_RESULTS), 'source_sha256': source_hash(),
    'reader': base_data['reader'],
    'prompt_sha256': {name: hashlib.sha256(text.encode()).hexdigest() for name, text in prompt_texts.items()},
    'conditions': [job['name'] for job in JOBS],
    'sole_prompt_change': 'Remove the exact abstention instruction; all other prompt text is identical.',
}
if directory_sha is not None:
    manifest['calibration_directory_sha256'] = directory_sha
save_json(OUTPUT_ROOT/'experiment_manifest.json', manifest)
print('Conditions:', *manifest['conditions'], sep='\n  ')
print('Planned reader calls:', len(sample_ids)*len(JOBS))


## Prepare Raw and Oracle contexts on CPU
Only the explicit Oracle construction receives gold supporting paragraph annotations. Neither baseline initializes an encoder or controller. Saved IterCOMP contexts are passed directly to the reader, without recompression.


In [ ]:
from contextlib import redirect_stdout
from itercomp.experiment import run_compression

CONTEXT_FILES = {'itercomp': COMPRESSION_RESULTS}
for method in ('raw', 'oracle'):
    directory = OUTPUT_ROOT/'contexts'/method
    log_path = OUTPUT_ROOT/f'prepare_{method}.log'
    with log_path.open('a', encoding='utf-8') as log, redirect_stdout(log):
        path = run_compression(
            config_from_dict(base_data), INPUTS, directory, method=method,
            gold_path=GOLD if method == 'oracle' else None,
            resume=(directory/'run_metadata.json').exists())
    CONTEXT_FILES[method] = Path(path)
# Verify every context/config pair before downloading model weights.
for job in JOBS:
    from itercomp.config import load_config
    cfg = load_config(job['config'])
    inspect_artifacts(CONTEXT_FILES[job['context']], samples, cfg, dataset_sha, prompt_hashes(load_prompts(cfg)))
print('Three context conditions verified; no BGE/controller inference was run.')


## Reader GPU and Hugging Face access
Only the Llama reader is checked or loaded. The token must belong to an account approved for this exact checkpoint. In Kaggle's Secrets panel, tick **HF_TOKEN** to enable access. The credential is inherited by reader subprocesses and is never written to the export.


In [ ]:
# Check native bitsandbytes CUDA support in a fresh process.
subprocess.run([sys.executable, '-c',
    "import torch, bitsandbytes as bnb; from bitsandbytes.cextension import lib; "
    "assert torch.cuda.is_available(), 'Enable a Kaggle GPU runtime'; "
    "assert lib is not None and getattr(lib, 'compiled_with_cuda', False), 'bitsandbytes CUDA backend missing'; "
    "print('torch', torch.__version__, 'CUDA', torch.version.cuda, 'bitsandbytes', bnb.__version__)"
], check=True)
from huggingface_hub import HfApi, get_token, get_hf_file_metadata, hf_hub_url
token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    pass
token = token.strip() if isinstance(token, str) else get_token()
if not token:
    raise RuntimeError('Enable the HF_TOKEN Kaggle secret for the approved Hugging Face account')
os.environ['HF_TOKEN'] = token
os.environ['HF_HUB_DISABLE_IMPLICIT_TOKEN'] = '0'
HfApi(token=token).whoami()
reader = base_data['reader']
get_hf_file_metadata(hf_hub_url(reader['model_id'], 'config.json', revision=reader['revision']), token=token)
del token
print('Reader authentication and checkpoint configuration access verified.')


## Run all six reader conditions
This is the GPU inference cell: **900 reader calls** for 150 questions × six conditions. Models load sequentially and are released when each subprocess exits. Rerunning resumes the same runs; use a different `OUTPUT_ROOT` for a different experiment.

A subprocess crash stops the cell and leaves checkpoints/logs available. An automatic checkpoint ZIP is saved after each condition and on failure. The normal runner records sample-level errors; the comparison will report them rather than omit those questions.


In [ ]:
def checkpoint_export():
    return shutil.make_archive(str(OUTPUT_ROOT.parent/(OUTPUT_ROOT.name+'_export')), 'zip', OUTPUT_ROOT)

def execute_logged(command, log_path):
    with Path(log_path).open('a', encoding='utf-8') as log:
        proc = subprocess.Popen(command, cwd=REPO, env=os.environ.copy(),
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, encoding='utf-8', errors='replace', bufsize=1)
        try:
            for line in proc.stdout:
                log.write(line)
                log.flush()
                if line.startswith('read '):
                    progress = line.split()[1].split('/')
                    if int(progress[0]) % 25 == 0 or progress[0] == progress[1]:
                        print(line.strip(), flush=True)
            status = proc.wait()
        finally:
            proc.stdout.close()
            if proc.poll() is None:
                proc.terminate()
                proc.wait()
        if status:
            raise RuntimeError(f'Reader subprocess exited {status}; inspect {log_path} and rerun this cell to resume.')

for job in JOBS:
    print('\nRunning:', job['name'], flush=True)
    command = [sys.executable, str(REPO/'scripts/run_experiment.py'),
               '--config', str(job['config']), '--inputs', str(INPUTS), '--split', 'calibration',
               '--stage', 'read', '--compression-results', str(CONTEXT_FILES[job['context']]),
               '--output-dir', str(job['output'])]
    if (job['output']/'run_metadata.json').exists():
        command.append('--resume')
    try:
        execute_logged(command, OUTPUT_ROOT/(job['name']+'.log'))
        meta = json.loads((job['output']/'run_metadata.json').read_text(encoding='utf-8'))
        if not meta.get('selected_subset_complete') or meta.get('completed_count') != len(sample_ids):
            raise RuntimeError(f"Incomplete reader condition: {job['name']}")
        actual_revision = meta.get('component_initialization', {}).get('reader', {}).get('resolved_revision')
        if actual_revision != reader_revision:
            raise ValueError(f"Resolved reader revision differs in {job['name']}: {actual_revision}")
        print('Completed:', job['name'], '| sample errors:', len(meta.get('failed_sample_ids', [])))
    finally:
        print('Checkpoint export:', checkpoint_export(), flush=True)


## Score every condition on the same calibration questions
Evaluation is CPU-only and uses the existing answer metrics and diagnostics. Existing reports are reused only when their prediction and gold hashes still match. Failed samples remain in the denominator.


In [ ]:
for job in JOBS:
    predictions = job['output']/'predictions.jsonl'
    meta = json.loads((job['output']/'run_metadata.json').read_text(encoding='utf-8'))
    if not meta.get('selected_subset_complete') or file_sha(predictions) != meta['output_sha256']:
        raise ValueError(f"Incomplete or altered predictions: {job['name']}")
    report = job['output']/'reports'
    evaluation_meta = report/'evaluation_metadata.json'
    if (report/'diagnostics/diagnostic_summary.json').exists() and (report/'scored/summary.json').exists():
        previous = json.loads(evaluation_meta.read_text(encoding='utf-8'))
        if (previous['predictions_sha256'] != file_sha(predictions)
                or previous['source_gold_sha256'] != file_sha(GOLD) or not previous['complete_split']):
            raise ValueError(f"Stale reports: {report}")
    else:
        # Preserve any interrupted report directory and evaluate into a fresh one.
        if report.exists() and any(report.iterdir()):
            from datetime import datetime, timezone
            report.rename(report.with_name('reports_incomplete_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%f')))
        subprocess.run([sys.executable, str(REPO/'scripts/evaluate_run.py'),
                        '--gold', str(GOLD), '--predictions', str(predictions), '--out', str(report)],
                       cwd=REPO, check=True)
print('All six reports are ready.')


## Compare performance, abstention, and paired changes
The summary shows EM/F1 on a 0–100 scale. Paired deltas compare the prompt variants question by question within each fixed context. Voluntary refusals expressed with other wording are retained as ordinary answers; the raw generations are available for inspection.

The archived IterCOMP/current predictions are also compared with the fresh control. Differences can reflect runtime or numerical variation and should be investigated before treating small changes as meaningful. Use calibration to select settings, then freeze them before another evaluation run; evaluation has already been inspected, so describe subsequent results transparently.


In [ ]:
from benchmark import answer_scores
gold_by_id = {row['id']: row for row in read_jsonl(GOLD)}
all_rows, table, per_sample = {}, [], []
for job in JOBS:
    rows = {row['id']: row for row in read_jsonl(job['output']/'predictions.jsonl')}
    if set(rows) != set(sample_ids):
        raise ValueError(f"ID coverage differs: {job['name']}")
    all_rows[job['name']] = rows
    scores = {}
    for rid in sample_ids:
        row, gold = rows[rid], gold_by_id[rid]
        em, f1 = answer_scores(row['predicted_answer'], [gold['answer'], *gold['answer_aliases']])
        abstain = not row['predicted_answer'].strip() and not row.get('error')
        scores[rid] = (em, f1, abstain)
        per_sample.append({'condition': job['name'], 'id': rid, 'hop': len(gold['question_decomposition']),
                           'gold_answer': gold['answer'], 'predicted_answer': row['predicted_answer'],
                           'reader_raw_response': row.get('reader_raw_response'),
                           'em': em, 'f1': f1, 'abstain': abstain, 'error': bool(row.get('error'))})
    job['scores'] = scores
    official = json.loads((job['output']/'reports/scored/summary.json').read_text(encoding='utf-8'))['overall']
    em = statistics.fmean(v[0] for v in scores.values())
    f1 = statistics.fmean(v[1] for v in scores.values())
    if abs(em-official['answer_em']) > 1e-12 or abs(f1-official['answer_f1']) > 1e-12:
        raise ValueError('Comparison scores disagree with saved reports')
    table.append({'condition': job['name'], 'n': len(rows), 'EM': 100*em, 'F1': 100*f1,
                  'abstention_percent': 100*statistics.fmean(v[2] for v in scores.values()),
                  'errors': sum(bool(r.get('error')) for r in rows.values()),
                  'mean_reader_seconds': statistics.fmean(r['reader_seconds'] for r in rows.values())})

paired = []
for context in ('raw', 'oracle', 'itercomp'):
    left = next(j for j in JOBS if j['context']==context and j['prompt']=='current')['scores']
    right = next(j for j in JOBS if j['context']==context and j['prompt']=='no_abstention_instruction')['scores']
    paired.append({'context': context,
                   'delta_EM_points': 100*statistics.fmean(right[r][0]-left[r][0] for r in sample_ids),
                   'delta_F1_points': 100*statistics.fmean(right[r][1]-left[r][1] for r in sample_ids),
                   'exact_matches_gained': sum(right[r][0] > left[r][0] for r in sample_ids),
                   'exact_matches_lost': sum(right[r][0] < left[r][0] for r in sample_ids),
                   'abstentions_removed_net': sum(left[r][2]-right[r][2] for r in sample_ids)})
archived = {row['id']: row for row in read_jsonl(ARCHIVED_PREDICTIONS)}
fresh = all_rows['itercomp_current']
replay = {
    'n': len(sample_ids),
    'predicted_answer_changes': sum(archived[r]['predicted_answer'] != fresh[r]['predicted_answer'] for r in sample_ids),
    'raw_generation_changes': sum(archived[r].get('reader_raw_response') != fresh[r].get('reader_raw_response') for r in sample_ids),
    'note': 'Same resolved checkpoint, settings and contexts; platform numerical differences can affect greedy decoding.'
}
comparison = {'conditions': table, 'paired_prompt_changes': paired, 'archived_control_comparison': replay}
# Reports are deterministic functions of the saved predictions and may be regenerated.
(OUTPUT_ROOT/'comparison.json').write_text(json.dumps(comparison, indent=2)+'\n', encoding='utf-8')
for filename, rows in [('comparison.csv', table), ('paired_changes.csv', paired), ('per_sample_comparison.csv', per_sample)]:
    with (OUTPUT_ROOT/filename).open('w', encoding='utf-8', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
print(f"{'Condition':<42} {'N':>4} {'EM':>8} {'F1':>8} {'Abstain%':>10} {'Errors':>7}")
for row in table:
    print(f"{row['condition']:<42} {row['n']:>4} {row['EM']:>8.2f} {row['F1']:>8.2f} {row['abstention_percent']:>10.2f} {row['errors']:>7}")
print('\nPaired prompt changes (no abstention instruction minus current):')
print(json.dumps(paired, indent=2))
print('\nArchived control comparison:', replay)


## Export results
Download the ZIP from Kaggle's output browser or save a notebook version with outputs. The export includes predictions, raw generations, per-condition reports/logs, both prompts, pinned configs, provenance, and paired comparisons. Checkpoint caches and credentials are not included.


In [ ]:
archive_path = checkpoint_export()
print('Export:', archive_path)
print('Compare: comparison.csv, paired_changes.csv, per_sample_comparison.csv')
print('Keep this archive together with the original calibration compression export.')
